In [1]:
import distopf as opf

## Run a power flow on the IEEE 13 bus system with no control variables.

In [2]:
case = opf.create_case(opf.CASES_DIR / "csv" / "ieee13")
result = case.run_pf()  # run power flow
result.plot_network()

## Run a power flow on the IEEE 123 bus system with no control variables.

In [3]:
case = opf.create_case(opf.CASES_DIR / "csv" / "ieee123")
result = case.run_pf()  # run power flow
result.plot_network()

# Run using OpenDSS Model

In [4]:
case = opf.create_case(opf.CASES_DIR / "dss" / "ieee123_dss" / "Run_IEEE123Bus.DSS")
result = case.run_pf()
result.plot_network()

## Run a power flow on the IEEE 123 bus system with DERs on 30 buses.
Give the DERs 10x power to demonstrate the effect on voltages.

In [5]:
case = opf.create_case(opf.CASES_DIR / "csv" / "ieee123_30der", ignore_schedule=True)
case.modify(gen_mult=10)
result = case.run_pf()
result.plot_network(v_max=1.05, v_min=0.95)

In [6]:
result.plot_voltages()

Now we will run the same power flow, but with curtailment minimization to bring the voltages in bounds.

In [13]:
result = case.run_opf(objective_function="curtail_min", control_variable="P")
result.plot_network()

In [8]:
result.plot_voltages()

In [9]:
result.plot_gens()

# Using plot_network
(latitude and longitude data required in bus_data.csv)

In [10]:
result.plot_network(show_phases="a", show_reactive_power=True)

In [11]:
result.plot_network(show_phases="b", show_reactive_power=False)

# Run using lower level API

In [12]:
case = opf.create_case(opf.CASES_DIR / "csv" / "ieee123_30der", ignore_schedule=True)
case.modify(control_variable="P", gen_mult=10)
model = opf.LinDistModel(
    branch_data=case.branch_data,
    bus_data=case.bus_data,
    gen_data=case.gen_data,
    cap_data=case.cap_data,
    reg_data=case.reg_data,
)
# Solve model using provided objective function
# result = opf.lp_solve(model, np.zeros(model.n_x))
result = opf.cvxpy_solve(model, opf.cp_obj_curtail)
print(result.fun)
v = model.get_voltages(result.x)
s = model.get_apparent_power_flows(result.x)
p = model.get_p_flows(result.x)
q = model.get_q_flows(result.x)
p_gens = model.get_p_gens(result.x)
q_gens = model.get_q_gens(result.x)
opf.plot_network(model, v, s, p_gen=p_gens, q_gen=q_gens).show()
opf.plot_voltages(v).show()
opf.plot_power_flows(p, q).show()

0.48013883015073655


AttributeError: 'DataFrame' object has no attribute 'voltages'